<a href="https://colab.research.google.com/github/Yusef-Hazem/Lang-Frameworks/blob/main/03Routing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/Langchain/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/Langchain


In [ ]:
# #Python 3.12.13
# !pip install -U langchain==1.3.11
# !pip install requests==2.34.2
# !pip install -U langchain_community==0.4.2
# !pip install -U python-dotenv==1.2.2
# !pip install -U langchain-huggingface==1.2.2
# !pip install faiss-cpu==1.14.3
# !pip install langchain-groq==1.1.3
# !pip install youtube-transcript-api==1.2.4
# !pip install pytube==15.0.0

In [ ]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
import bs4
from dotenv import load_dotenv, find_dotenv
import os
load_dotenv(find_dotenv())

/tmp/ipykernel_9578/3427602061.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader


True

In [ ]:
# set enviroment variable of the groq APi key
# gsk_DkOInQs0mmiT7z2dlNNBWGdyb3FY021sV6oCN80L55hpKbdX7Q9vxx
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

Routing :
- `Logical routing :` let the llm choose database based-on question
- `Semantic routing :` embed question and choose prombt based on similarity

In [ ]:
from pydantic import BaseModel  , Field
from typing import Literal
class RouteQuery(BaseModel) :
  datasource : Literal["python_docs", "js_docs", "golang_docs" ] = Field(...,
                                                  description= "these are the datasourses you should choose based-on user question  ")


llm= ChatGroq(model="Llama-3.3-70B-Versatile", temperature=0)
llm_structured = llm.with_structured_output(RouteQuery)




In [ ]:
# prepare the prompt

system = """You are an expert at routing a user question to the appropriate data source.

Based on the programming language the question is referring to, route it to the relevant data source."""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "{question}"),
    ]
)

In [ ]:
llm_structured.invoke("What is the mean of decomposion concept in programming ?")

RouteQuery(datasource='python_docs')

In [ ]:
# pass prompt to the llm_structured


logical_routing_chain = (
    prompt
    | llm_structured
    | (lambda x : x.datasource)
    | StrOutputParser()
)

question = "What is the mean of decomposion concept in programming ? "
logical_routing_chain.invoke({"question" : question})

'python_docs'

Semantic Routing
- choose the prompt based on the question , embed each prompt and make comparison between it and the question

In [ ]:
# you have two prompts you want to use each in there place
physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise and easy to understand manner. \
When you don't know the answer to a question you admit that you don't know.

Here is a question:
{query}"""

math_template = """You are a very good mathematician. You are great at answering math questions. \
You are so good because you are able to break down hard problems into their component parts, \
answer the component parts, and then put them together to answer the broader question.

Here is a question:
{query}"""

In [ ]:
model_name = "BAAI/bge-small-en"
model_kwargs = {"device": "cpu"}
encode_kwargs = {"normalize_embeddings": True}
embedding_model = HuggingFaceEmbeddings(
    model_name=model_name, model_kwargs=model_kwargs, encode_kwargs=encode_kwargs
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
prompt_templates = [physics_template , math_template]
embedded_queries = embedding_model.embed_documents(prompt_templates)


In [ ]:
from langchain_community.utils.math import cosine_similarity
def choose_prompt (question) :
  question_embedding = embedding_model.embed_query(question)

  similarity = cosine_similarity([question_embedding] ,embedded_queries )[0]
  most_similar = prompt_templates[similarity.argmax()]
  print("using_math" if most_similar == math_template else "using_pythics")
  return ChatPromptTemplate.from_template(most_similar)

In [ ]:
from langchain_core.runnables import RunnableLambda

which_prompt_to_select_chain = (RunnableLambda(choose_prompt) |
                                  llm |
                                  StrOutputParser() )

which_prompt_to_select_chain.invoke("what is the black hole")

using_pythics


"A black hole is a region in space where the gravitational pull is so strong that nothing, including light, can escape. It's formed when a massive star collapses in on itself, causing a massive amount of matter to be compressed into an incredibly small space, creating an intense gravitational field.\n\nImagine a super-powerful vacuum cleaner that warps the fabric of space and time around it, pulling everything in and not letting it out. That's basically what a black hole is. The point of no return, called the event horizon, marks the boundary of the black hole. Once you cross the event horizon, you're trapped, and the gravity is so strong that you'll be pulled towards the center, known as the singularity.\n\nBut here's the really cool thing about black holes: they come in different sizes, from small, stellar-mass black holes formed from star collapses, to supermassive black holes found at the centers of galaxies, with masses millions or even billions of times that of our sun.\n\nWould 